In [1]:
!pip install -q google-genai

In [2]:
import os
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")

print("API key loaded successfully.")

Enter your Gemini API key: ··········
API key loaded successfully.


In [5]:

import os
import re
import pandas as pd
from google import genai

# Initialize Gemini
API_KEY = os.environ["GEMINI_API_KEY"]
MODEL = "gemini-3.6-flash"

client = genai.Client(api_key=API_KEY)

# 5 test reviews and ground-truth labels
reviews = [
    ("The battery life is amazing!", "Positive"),
    ("It works, nothing special.", "Neutral"),
    ("The phone stopped working after one day.", "Negative"),
    ("The camera takes beautiful pictures.", "Positive"),
    ("The screen is too small.", "Negative")
]

# Extract exactly 5 sentiment labels from the model response
def get_labels(response_text):
    labels = re.findall(
        r"\b(Positive|Neutral|Negative)\b",
        response_text,
        re.IGNORECASE
    )

    labels = [label.capitalize() for label in labels]

    if len(labels) != 5:
        raise ValueError(
            f"Expected 5 labels but received {len(labels)}. "
            f"Model response: {response_text}"
        )

    return labels


# Make one API call for all five reviews
def classify_all(prompt):
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"temperature": 0}
    )

    if not response.text:
        raise ValueError("The model returned an empty response.")

    return get_labels(response.text)


# Prepare test reviews
test_reviews = "\n".join(
    f"{i+1}. {review}"
    for i, (review, _) in enumerate(reviews)
)

# Zero-shot prompt: no examples
zero_prompt = f"""
Classify each customer review as Positive, Neutral, or Negative.

Return exactly five lines in this format:
1. Positive
2. Neutral
3. Negative
4. Positive
5. Negative

Do not include explanations or repeat the examples.

Reviews:
{test_reviews}
"""

# Few-shot prompt: three examples
few_prompt = f"""
Classify each customer review as Positive, Neutral, or Negative.

Examples:
Review: "Best phone ever!" -> Positive
Review: "Average quality, nothing special." -> Neutral
Review: "The camera is broken." -> Negative

Now classify the following five reviews.
Return exactly five numbered lines, with only the sentiment label
after each number. Do not include explanations.

Reviews:
{test_reviews}
"""

print("Running zero-shot classification...")
zero_labels = classify_all(zero_prompt)

print("Running few-shot classification...")
few_labels = classify_all(few_prompt)

# Compare results
rows = []

for i, (review, actual) in enumerate(reviews):
    zero = zero_labels[i]
    few = few_labels[i]

    rows.append({
        "Review": review,
        "Actual": actual,
        "Zero-Shot": zero,
        "Few-Shot": few,
        "Zero Correct": zero == actual,
        "Few Correct": few == actual,
        "Same Prediction": zero == few,
        "Zero Format": zero in ["Positive", "Neutral", "Negative"],
        "Few Format": few in ["Positive", "Neutral", "Negative"]
    })

df = pd.DataFrame(rows)

print("\nZERO-SHOT VS FEW-SHOT RESULTS")
display(df)

# Comparison summary
print("\nCOMPARISON SUMMARY")
print("-" * 40)

print(
    f"Zero-shot accuracy: "
    f"{df['Zero Correct'].sum()}/5 "
    f"({df['Zero Correct'].mean() * 100:.1f}%)"
)

print(
    f"Few-shot accuracy: "
    f"{df['Few Correct'].sum()}/5 "
    f"({df['Few Correct'].mean() * 100:.1f}%)"
)

print(
    f"Zero-shot format adherence: "
    f"{df['Zero Format'].sum()}/5"
)

print(
    f"Few-shot format adherence: "
    f"{df['Few Format'].sum()}/5"
)

print(
    f"Agreement between methods: "
    f"{df['Same Prediction'].sum()}/5"
)

# Save results
df.to_csv("sentiment_comparison.csv", index=False)
print("\nResults saved to sentiment_comparison.csv")


Running zero-shot classification...
Running few-shot classification...

ZERO-SHOT VS FEW-SHOT RESULTS


,Review,Actual,Zero-Shot,Few-Shot,Zero Correct,Few Correct,Same Prediction,Zero Format,Few Format
0,The battery life is amazing!,Positive,Positive,Positive,True,True,True,True,True
1,"It works, nothing special.",Neutral,Neutral,Neutral,True,True,True,True,True
2,The phone stopped working after one day.,Negative,Negative,Negative,True,True,True,True,True
3,The camera takes beautiful pictures.,Positive,Positive,Positive,True,True,True,True,True
4,The screen is too small.,Negative,Negative,Negative,True,True,True,True,True



COMPARISON SUMMARY
----------------------------------------
Zero-shot accuracy: 5/5 (100.0%)
Few-shot accuracy: 5/5 (100.0%)
Zero-shot format adherence: 5/5
Few-shot format adherence: 5/5
Agreement between methods: 5/5

Results saved to sentiment_comparison.csv
